# 🎬 Tormie - Magnet Link Downloader

**Simpel:** Paste magnet link → Download → Tersimpan di Google Drive

Jalankan cell dari atas ke bawah.

In [ ]:
#@title 1. Setup (Jalankan sekali)
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Buat folder download
import os
DOWNLOAD_DIR = '/content/drive/MyDrive/Tormie_Movies'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Install libtorrent
!apt-get install -qq python3-libtorrent -y 2>/dev/null

print('\u2705 Setup selesai! Folder:', DOWNLOAD_DIR)

In [ ]:
#@title 2. Paste Magnet Link & Download
import libtorrent as lt
import time
from IPython.display import clear_output

# --- PASTE MAGNET LINK DI SINI ---
MAGNET_LINK = input('🧲 Paste magnet link: ')

if not MAGNET_LINK.startswith('magnet:'):
    print('\u274c Bukan magnet link! Harus dimulai dengan magnet:')
else:
    # Setup session
    ses = lt.session()
    ses.listen_on(6881, 6891)
    
    params = lt.parse_magnet_uri(MAGNET_LINK)
    params.save_path = DOWNLOAD_DIR
    handle = ses.add_torrent(params)
    
    print('⏳ Mengambil metadata...')
    while not handle.has_metadata():
        time.sleep(1)
    
    info = handle.get_torrent_info()
    print(f'\u2705 {info.name()}')
    print(f'📦 Size: {info.total_size() / (1024**3):.2f} GB')
    print(f'📁 Save to: {DOWNLOAD_DIR}')
    print('\n⬇\ufe0f Downloading...\n')
    
    # Download loop
    while not handle.is_seed():
        s = handle.status()
        progress = s.progress * 100
        down = s.download_rate / 1e6  # MB/s
        peers = s.num_peers
        
        bar_len = 30
        filled = int(bar_len * s.progress)
        bar = '\u2588' * filled + '\u2591' * (bar_len - filled)
        
        clear_output(wait=True)
        print(f'🎬 {info.name()}')
        print(f'[{bar}] {progress:.1f}%')
        print(f'\u2b07\ufe0f {down:.2f} MB/s | 👥 {peers} peers')
        
        time.sleep(3)
    
    clear_output(wait=True)
    print(f'\u2705 SELESAI! {info.name()}')
    print(f'📁 Tersimpan di: {DOWNLOAD_DIR}/{info.name()}')

In [ ]:
#@title 3. Cek File (Opsional)
import os

print(f'📁 {DOWNLOAD_DIR}:\n')
for item in sorted(os.listdir(DOWNLOAD_DIR)):
    path = os.path.join(DOWNLOAD_DIR, item)
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024**2)
        print(f'  🎬 {item} ({size:.0f} MB)')
    else:
        print(f'  📂 {item}/')